# xNES versus default CMA-ES: noisy BBOB landscapes

This study measures **the clean objective at the current mean** versus objective calls. Both optimizers see only noisy scores; oracle assessment never enters an update or stopping decision.

- Bare xNES and default pycma populations, identical zero start and isotropic scale 1.
- No wrapper resets, external restarts, explicit noise handling, or learning-rate tuning.
- Known BBOB landscapes with controlled noise—not the official `bbob-noisy` suite. [Function definitions](https://numbbo.github.io/coco/testsuites/bbob).
- Preview is exploratory; use the full preset for 15 instances and two optimizer seeds per case. No experiment can establish superiority outside its tested budgets and noise models.
- Every terminal status, warning, and exception is retained. A convergence-like stop can still have poor objective quality.

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys

# Works from either the repository root or notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "leitwerk").is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from IPython.display import display
from benchmarks.runner import PRESETS, CHECKPOINTS, create_run, run_study, run_stability, load_runs
from benchmarks.reporting import checkpoint_table, paired_summary, run_table, plot_quality, plot_targets, plot_stability

In [ ]:
# Run All launches the preview, never the full study implicitly.
PRESET = "preview"  # "full": 24 functions, d=2/5/10/20/30, 15 instances, two seeds, 10,000 calls
config = PRESETS[PRESET]
# Optional edits, e.g. config = replace(config, functions=(12, 15), dimensions=(10, 30))
LOAD_RUN = None  # Path("runs/<saved directory>") to regenerate reports without optimization
print({
    "preset": PRESET, "config": config,
    "optimizer_runs": config.runs,
    "maximum_objective_calls": config.runs * config.budget,
    "checkpoints": [n for n in CHECKPOINTS if n <= config.budget],
})
print("Stability runs are separate; the full preset can take a long time.")

In [ ]:
# Noise definitions (s = max(initial clean gap, 1e-12); g = current clean gap):
# clean:           f
# additive:        f + 0.01*s*N(0,1)
# heteroscedastic: f + (0.1*g + 0.001*s)*N(0,1)
# outliers:        f + 0.01*s*N(0,1) + Bernoulli(0.01)*s*t(3)
#
# Random variates are paired by evaluation index, not by generation.
# Each optimizer uses its native population size. Only complete populations are told.
# The final remainder is unused; checkpoints use the latest completed update, never a future one.
#
# Clean assessment is outside the optimization budget and never consumes observation noise.
# Budgets count all attempted optimizer objective calls, including calls producing invalid values.
# The core's first terminal status ends the run; target attainment is assessed afterward.

In [ ]:
if LOAD_RUN is None:
    run_dir = create_run(config)
    print("Writing incremental results to", run_dir)
    results = run_study(config, run_dir)
else:
    run_dir = Path(LOAD_RUN)
    results = load_runs(run_dir)
    print("Loaded", len(results), "runs from", run_dir)

runs = run_table(results)
print(f"Recorded run time: {runs.seconds.sum():.1f} s; objective calls: {runs.evaluations.sum():,}")
display(runs.groupby(["algorithm", "kind"], dropna=False).agg(
    runs=("problem", "size"), calls=("evaluations", "sum"), warnings=("warning_count", "sum")
))

In [ ]:
# Every failure/early stop remains visible; inspect a run's JSON for its saved pre-update state.
display(runs[runs.kind != "budget"][
    ["function", "dimension", "instance", "repeat", "noise", "algorithm",
     "kind", "reason", "evaluations", "unused_budget", "final_relative_gap", "warning_count"]
])
display(runs[runs.warning_count > 0][
    ["function", "dimension", "noise", "algorithm", "warning_messages"]
])
print("Numerical failures use their last finite state only for diagnosis, not as successful recommendations.")

In [ ]:
checkpoints = checkpoint_table(results)
comparison = paired_summary(checkpoints)
# Positive median_log10_gap_ratio favors CMA; negative favors xNES.
# Intervals resample whole instances, retaining their repeated seeds together.
# They describe uncertainty of paired differences, not the spread of unrelated cases.
display(comparison)
print("Finite-only estimates must be read with excluded_pairs and each algorithm's failure count.")
print("Preview intervals use only three instances and should not support strong claims.")

In [ ]:
# Solid: current-mean median; band: case IQR (NOT a confidence interval).
# Dotted: best clean evaluated gap, an oracle diagnostic rather than a deployable recommendation.
# Finite stopped recommendations are carried forward and marked in the checkpoint table.
# Failed runs leave finite-quality curves at failure; failure counts are shown and retained in tables.
plot_quality(results)
plt.show()
# To inspect one landscape: plot_quality(results, function=12)

In [ ]:
# First attainment is not sustained convergence; a noisy trajectory can deteriorate afterward.
# All run-target pairs, including unreached targets and failures, remain in the denominator.
plot_targets(results)
plt.show()

## Stability without recovery

Constant scores exercise actual tie handling. Independent random scores exercise random rankings. Neither contains optimization signal: drift is measured, not judged against an invented threshold. Initial scales are randomly rotated with geometric-mean scale 1 and axis ratios 1, 10⁶, and 10⁹. Dimensions are 2, 10, and 30.

The first terminal status ends each run. No exception is swallowed as a success. Diagnostic JSON files contain the pre-update distribution, population, scores, and xNES standardized samples needed to replay a failed update.

In [ ]:
if LOAD_RUN is None:
    stability = run_stability(config, run_dir)
else:
    stability = load_runs(run_dir, "stability-*.json")
stability_table = run_table(stability)
display(stability_table[[
    "dimension", "initial_axis_ratio", "repeat", "noise", "kind", "reason", "evaluations",
    "max_mean_drift", "max_axis_ratio", "final_scale_global", "warning_count"
]])
plot_stability(stability)
plt.show()

## Reading and reproducing the results

Every run is written immediately under a unique `runs/` directory; nothing is deleted. The manifest records configuration, seeds (also per run), package versions, current xNES learning-rate defaults, revision, and source hashes. JSON includes complete generation diagnostics and terminal warnings/errors.

Set `LOAD_RUN` above to regenerate these reports. When comparing revisions, use identical problem/noise configurations. These experiments establish behavior only for the specified landscapes, noise distributions, dimensions, and budgets—not a universal ranking. The workbench is for quick experiments; parameter sweeps are training experiments and should not be used as held-out evidence.